# Module 6: GPU Memory Hierarchy & The PCIe Bottleneck

Welcome! In this module, we will explore the GPU memory layout and see why copying data to the GPU is a major operation.

**Section Goals:**
* Understand the difference between CPU RAM and GPU VRAM.
* Learn about the PCIe bus bottleneck.
* Observe the difference between transfer cost and math cost.

### The PCIe Suspension Bridge

The GPU has its own ultra-fast memory built directly on its card, called **VRAM (Video RAM)** or **HBM (High Bandwidth Memory)**. 

However, the GPU is connected to the CPU and System RAM via a hardware pathway called the **PCIe slot**.

Think of PCIe as a **narrow suspension bridge** connecting two islands. Moving cargo (tensors) across the bridge is slow. But once the cargo is on the GPU island, delivery trucks (HBM) move it around almost instantly.

### Visualizing the PCIe Bridge

Here is a whiteboard sketch representing the slow transfer link versus fast internal GPU memory access:

![PCIe Suspension Bridge](images/pcie-suspension-bridge.svg)

### Step 1: Measuring Transfer Cost (The Bridge Cross)

Let's write a script to measure how long it takes to transfer a tensor of 10,000,000 elements from CPU RAM to GPU VRAM.

In [ ]:
x_cpu = torch.randn(10000000, device="cpu")
torch.cuda.synchronize()  # Clear GPU stream
t0 = time.perf_counter()  # Start CPU timer
x_gpu = x_cpu.to("cuda")  # Move data over PCIe to GPU
torch.cuda.synchronize()  # Wait for transfer completion
print(f"Transfer time: {time.perf_counter() - t0:.4f}s")

### Step 2: Measuring GPU Math Cost (On-Island speed)

Now that the tensor is already on the GPU island, let's time a math operation on it.

In [ ]:
torch.cuda.synchronize()  # Clear GPU stream
t0 = time.perf_counter()  # Start CPU timer
y_gpu = x_gpu + 1.0  # Run math on GPU
torch.cuda.synchronize()  # Wait for completion
print(f"GPU Math time: {time.perf_counter() - t0:.6f}s")

### Interpretation

The data transfer took significantly longer than the actual math calculation. This is the **Memory Wall** in action! Moving data to the GPU is very slow. If you only do a simple operation on it, the transfer cost is not worth it. 

Rule of thumb: **Move data to the GPU once, and run many computations on it!**

### Module 6 Recap

* **VRAM/HBM** is the GPU's local, ultra-fast memory.
* The **PCIe Bus** is a slow bridge connecting CPU RAM and GPU VRAM.
* Avoid copying data back and forth; keep tensors on the GPU as much as possible.